# ACCIDENT @ CVPR: Zero-Shot CCTV Traffic Accident Understanding

**Competition:** ACCIDENT @ CVPR 2026 -- AUTOPILOT Workshop (Kaggle)

**Notebook Focus:**
- Temporal localization of accident onset from fixed-view CCTV video clips
- Spatial localization of impact point via open-vocabulary grounding and optical flow
- Zero-shot collision-type classification using pre-trained vision-language models

## Introduction

Traffic accident analysis from fixed CCTV infrastructure presents challenges distinct from general video understanding. Accident events are temporally sparse, spatially unpredictable, and visually subtle relative to normal traffic flow. The benchmark defined by ACCIDENT @ CVPR 2026 compounds these challenges by excluding labeled real training footage, requiring methods that generalize without dataset-specific fine-tuning.

This notebook constructs a zero-shot inference pipeline whose **primary signal is per-vehicle detection and tracking**: an accident is an event between two specific objects, so the colliding pair's kinematics (contact time, joint deceleration, box-intersection centroid, approach-angle geometry) answer all three benchmark questions more directly than any whole-frame signal can. A chain of zero-shot vision-language models backs every stage up when tracking is unreliable:

| Stage | Question | Primary signal (tracking) | Fallback (zero-shot) |
|-------|----------|---------------------------|----------------------|
| 1 -- *When* | accident time | first box contact + joint deceleration peak | frame-diff anchor + bounded PE refinement |
| 2 -- *What* | collision type | pre-impact velocity angle + contact geometry | CLIP + Qwen2.5-VL soft ensemble |
| 3 -- *Where* | impact point | box-intersection centroid at impact | OWLv2 grounding + flow centroid |

The synthetic CARLA dataset is used for pipeline calibration and EDA. Submissions target the real CCTV test set. Every design decision below is validated (or rejected) on a *diverse* calibration set covering all five collision types -- an earlier revision tuned on a head-on-only subset and regressed badly on the other four types (see Section 7).

## Table of Contents

1. [Data Acquisition](#1-data-acquisition)
2. [Data Inspection](#2-data-inspection)
3. [Data Cleaning](#3-data-cleaning)
4. [Exploratory Data Analysis](#4-exploratory-data-analysis)
5. [Feature Engineering](#5-feature-engineering)
6. [Modeling](#6-modeling)
7. [Evaluation](#7-evaluation)
8. [Test Inference and Submission](#8-test-inference-and-submission)
9. [Conclusion](#9-conclusion)
10. [References](#10-references)

---
## 0. Environment Setup

In [1]:
# [SETUP] Standard library imports -- order: stdlib, third-party, local
import re
import json
import gzip
import time
import warnings
import pathlib
from collections import defaultdict

# [SETUP] Numerical and data processing
import numpy as np
import pandas as pd

# [SETUP] Visualization stack
import matplotlib.pyplot as plt
import seaborn as sns

# [SETUP] Computer vision
import cv2
from PIL import Image as PILImage

# [SETUP] Suppress non-critical runtime warnings
warnings.filterwarnings('ignore')

print('[STATUS] Core imports complete')

[STATUS] Core imports complete


In [2]:
# [SETUP] Global matplotlib and seaborn configuration
sns.set_theme(style='whitegrid', context='notebook')

plt.rcParams.update({
    'figure.figsize'  : (10, 6),
    'axes.titlesize'  : 13,
    'axes.labelsize'  : 11,
    'xtick.labelsize' : 10,
    'ytick.labelsize' : 10,
    'legend.fontsize' : 10,
    'figure.dpi'      : 120,
    'savefig.bbox'    : 'tight',
})

# [SETUP] Consistent color palette for all plots
PALETTE = {
    'primary'    : '#1f77b4',
    'secondary'  : '#ff7f0e',
    'tertiary'   : '#2ca02c',
    'quaternary' : '#d62728',
    'quinary'    : '#9467bd',
    'senary'     : '#8c564b',
}

print('[STATUS] Plot configuration applied')

[STATUS] Plot configuration applied


In [3]:
# [SETUP] Reproducibility seed for all stochastic operations
SEED = 42
np.random.seed(SEED)

# [SETUP] Root paths -- auto-detected, do not hardcode.
#
# Kaggle mounts data at a DIFFERENT prefix depending on how it was attached:
#   - competition data (Add Input -> Competition) -> /kaggle/input/competitions/<slug>/
#   - a regular Dataset (Add Input -> Dataset)     -> /kaggle/input/<slug>/
# A previous revision hardcoded the Dataset-style path
# (/kaggle/input/accident) with a comment noting the competition-style path
# was the correct one -- and never applied its own fix. Every downstream
# exists() check then silently returned False and produced 0 videos. Probe
# both known layouts, and if neither matches, search all of /kaggle/input for
# any directory that actually contains a sim_dataset/ or videos/ folder, so
# this survives being forked, renamed, or re-attached differently.
_CANDIDATE_ROOTS = [
    pathlib.Path('/kaggle/input/competitions/accident'),
    pathlib.Path('/kaggle/input/accident'),
]

def _looks_like_dataset_root(p: pathlib.Path) -> bool:
    return p.is_dir() and ((p / 'sim_dataset').exists() or (p / 'videos').exists())

BASE_DIR = next((r for r in _CANDIDATE_ROOTS if _looks_like_dataset_root(r)), None)

if BASE_DIR is None:
    _input_root = pathlib.Path('/kaggle/input')
    if _input_root.exists():
        for _entry in sorted(_input_root.iterdir()):
            if _looks_like_dataset_root(_entry):
                BASE_DIR = _entry
                break
            # competition-style datasets can nest one level deeper, e.g.
            # /kaggle/input/competitions/<slug>/
            if _entry.is_dir():
                for _sub in sorted(_entry.iterdir()):
                    if _looks_like_dataset_root(_sub):
                        BASE_DIR = _sub
                        break
            if BASE_DIR is not None:
                break

if BASE_DIR is None:
    print('[ERROR] Could not auto-detect the dataset root under /kaggle/input.')
    print('        Attach the "accident" competition data via Add Input, or if it is')
    print('        already attached, verify the actual layout below and set BASE_DIR')
    print('        manually in this cell.')
    _input_root = pathlib.Path('/kaggle/input')
    if _input_root.exists():
        print(f'[STATUS] Contents of {_input_root}:')
        for _entry in sorted(_input_root.iterdir()):
            print(f'    {_entry}')
    else:
        print('[STATUS] /kaggle/input does not exist -- not running on Kaggle, or no data attached.')
    BASE_DIR = pathlib.Path('/kaggle/input/accident')  # placeholder so later cells don't NameError

SYNTHETIC_DIR   = BASE_DIR / 'sim_dataset'    # synthetic CARLA dataset
REAL_VIDEOS_DIR = BASE_DIR / 'videos'         # real CCTV test videos
OUTPUT_DIR      = pathlib.Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# [SETUP] Key file paths
LABELS_CSV            = SYNTHETIC_DIR / 'labels.csv'                # synthetic supervision index
TEST_METADATA_CSV     = BASE_DIR      / 'test_metadata.csv'         # real test scene tags
SAMPLE_SUBMISSION     = BASE_DIR      / 'sample_submission.csv'     # output schema reference
ANNOTATION_CLASSES    = SYNTHETIC_DIR / 'annotation_classes.yaml'   # segmentation class map
SYNTHETIC_VIDEOS_DIR  = SYNTHETIC_DIR / 'videos'                    # {head-on,rear-end,sideswipe,single,t-bone}/
VIDEO_ANNOTATIONS_DIR = SYNTHETIC_DIR / 'video_annotations'         # per-video annotation files

COLLISION_TYPES = ['head-on', 'rear-end', 'sideswipe', 'single', 't-bone']


def resolve_video_path(rgb_path: str) -> pathlib.Path:
    """Resolve an rgb_path value from labels.csv to an absolute video path.

    rgb_path values are relative to sim_dataset/ (e.g. videos/head-on/clip.mp4),
    but earlier dataset revisions used paths relative to the competition root.
    Try both roots and return whichever exists, so downstream code never
    silently operates on a non-existent path (an earlier revision of this
    notebook lost 20/20 calibration videos to exactly that bug).
    """
    p = pathlib.Path(rgb_path)
    if p.is_absolute():
        return p
    for root in (SYNTHETIC_DIR, BASE_DIR):
        candidate = root / p
        if candidate.exists():
            return candidate
    return SYNTHETIC_DIR / p  # best guess; existence is asserted before inference


print('[STATUS] Path constants initialized')
print(f'  BASE_DIR        : {BASE_DIR}')
print(f'  SYNTHETIC_DIR   : {SYNTHETIC_DIR}')
print(f'  REAL_VIDEOS_DIR : {REAL_VIDEOS_DIR}')

[STATUS] Path constants initialized
  BASE_DIR        : /kaggle/input/competitions/accident
  SYNTHETIC_DIR   : /kaggle/input/competitions/accident/sim_dataset
  REAL_VIDEOS_DIR : /kaggle/input/competitions/accident/videos


In [4]:
# [STATUS] File availability audit -- all critical paths verified before downstream steps
path_checks = [
    ('BASE_DIR',              BASE_DIR),
    ('SYNTHETIC_DIR',         SYNTHETIC_DIR),
    ('SYNTHETIC_VIDEOS_DIR',  SYNTHETIC_VIDEOS_DIR),
    ('VIDEO_ANNOTATIONS_DIR', VIDEO_ANNOTATIONS_DIR),
    ('REAL_VIDEOS_DIR',       REAL_VIDEOS_DIR),
    ('labels.csv',            LABELS_CSV),
    ('annotation_classes.yaml', ANNOTATION_CLASSES),
    ('test_metadata.csv',     TEST_METADATA_CSV),
    ('sample_submission.csv', SAMPLE_SUBMISSION),
]

audit_df = pd.DataFrame([
    {
        'label'  : label,
        'path'   : str(p),
        'exists' : p.exists(),
        'kind'   : 'dir' if p.is_dir() else ('file' if p.is_file() else 'missing'),
    }
    for label, p in path_checks
])

# All rows should show exists=True before proceeding
display(audit_df.style.map(
    lambda v: 'color: green; font-weight: bold' if v is True
              else ('color: red; font-weight: bold' if v is False else ''),
    subset=['exists']
))

,label,path,exists,kind
0,BASE_DIR,/kaggle/input/competitions/accident,True,dir
1,SYNTHETIC_DIR,/kaggle/input/competitions/accident/sim_dataset,True,dir
2,SYNTHETIC_VIDEOS_DIR,/kaggle/input/competitions/accident/sim_dataset/videos,True,dir
3,VIDEO_ANNOTATIONS_DIR,/kaggle/input/competitions/accident/sim_dataset/video_annotations,True,dir
4,REAL_VIDEOS_DIR,/kaggle/input/competitions/accident/videos,True,dir
5,labels.csv,/kaggle/input/competitions/accident/sim_dataset/labels.csv,True,file
6,annotation_classes.yaml,/kaggle/input/competitions/accident/sim_dataset/annotation_classes.yaml,True,file
7,test_metadata.csv,/kaggle/input/competitions/accident/test_metadata.csv,True,file
8,sample_submission.csv,/kaggle/input/competitions/accident/sample_submission.csv,False,missing


In [5]:
# [SETUP] Install model dependencies (requires internet on Kaggle)
#   - ultralytics    : YOLOv8 vehicle detector (tracking stage)
#   - CLIP           : zero-shot classification (Stage 2)
#   - perception_models : Perception Encoder for temporal refinement (Stage 1)
#   - transformers stack : Qwen2.5-VL (Stage 2) and OWLv2 (Stage 3)
import subprocess
import sys
import json as _json


def _sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode, r.stdout, r.stderr


def _torch_smoke_test():
    """Import torch and run ONE real op on cuda, in a fresh subprocess.

    torch.cuda.is_available() returning True is not sufficient evidence the
    GPU actually works: a torch build whose compiled kernels don't cover this
    GPU's compute capability still reports CUDA as "available" and only fails
    once a real kernel launch is attempted. Also captures get_device_name /
    get_device_capability / get_arch_list, so an architecture mismatch is
    diagnosed directly instead of guessed at from the exception text alone.
    Runs in a subprocess -- not `import torch` in this cell -- because once
    this kernel process has torch in sys.modules, a later pip install that
    replaces the on-disk package would not change what an already-imported
    `torch` in this process reports.
    """
    code = (
        "import torch, json\n"
        "info = {'version': torch.__version__, 'cuda_available': torch.cuda.is_available()}\n"
        "try:\n"
        "    if info['cuda_available']:\n"
        "        info['device_name'] = torch.cuda.get_device_name(0)\n"
        "        info['device_capability'] = list(torch.cuda.get_device_capability(0))\n"
        "        info['torch_arch_list'] = torch.cuda.get_arch_list()\n"
        "        _ = (torch.tensor([1.0, 2.0], device='cuda') * 2).cpu()\n"
        "        _ = torch.prod(torch.tensor([2, 3, 4], dtype=torch.int64, device='cuda')).cpu()\n"
        "        info['cuda_op_ok'] = True\n"
        "    else:\n"
        "        info['cuda_op_ok'] = None\n"
        "except Exception as e:\n"
        "    info['cuda_op_ok'] = False\n"
        "    info['cuda_op_error'] = f'{type(e).__name__}: {e}'\n"
        "print(json.dumps(info))\n"
    )
    rc, out, err = _sh([sys.executable, '-c', code])
    lines = [l for l in out.strip().splitlines() if l.strip()]
    if lines:
        try:
            return _json.loads(lines[-1])
        except _json.JSONDecodeError:
            pass
    return {'version': None, 'cuda_available': None, 'cuda_op_ok': False,
            'raw_stdout': out[-500:], 'raw_stderr': err[-500:]}


def _diagnose_arch_mismatch(info):
    """Compare the GPU's actual compute capability against what this torch
    build shipped kernels for, so the report says WHY, not just THAT it failed."""
    cap = info.get('device_capability')
    arch_list = info.get('torch_arch_list') or []
    if not cap or not arch_list:
        return None
    sm = f'sm_{cap[0]}{cap[1]}'
    compiled_sms = {a.split('_')[0] + '_' + a.split('_')[1][:2] for a in arch_list if 'sm_' in a}
    if sm not in compiled_sms and not any(a.startswith(f'compute_{cap[0]}{cap[1]}') for a in arch_list):
        return (f"GPU is {info.get('device_name')} (compute capability {cap[0]}.{cap[1]}, i.e. {sm}), "
                f"but this torch build only has compiled kernels for: {arch_list}. "
                f"{sm} is NOT in that list -- this is an architecture mismatch, not a driver problem.")
    return None


print('[STATUS] Baseline torch smoke test (before any installs)...')
_baseline = _torch_smoke_test()
print(f'[STATUS] Baseline: {_baseline}')

_repaired = False
if _baseline.get('cuda_op_ok') is False:
    _diag = _diagnose_arch_mismatch(_baseline)
    if _diag:
        print(f'[DIAGNOSIS] {_diag}')
    else:
        print('[DIAGNOSIS] cuda_available=True but a real op fails, and device_capability/'
              'arch_list were not both captured (the failure happened before those calls) -- '
              'consistent with either an architecture mismatch or a driver/runtime version '
              'mismatch between this torch build and the host.')
    print('[WARNING] A real CUDA op already fails before any installs run -- this is a '
          'pre-existing problem with the environment\'s preinstalled torch, not something '
          'the installs below will cause. Attempting a repair: reinstalling torch/torchvision/'
          'torchaudio from the stable cu121 channel, which has shipped sm_75 (T4) kernels '
          'continuously and is a safer bet than whatever cu128 build shipped with this image.')
    _rc, _out, _err = _sh([
        sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
        'torch', 'torchvision', 'torchaudio',
        '--index-url', 'https://download.pytorch.org/whl/cu121',
    ])
    print(f'[STATUS] Repair install exit code: {_rc}')
    if _rc != 0:
        print(_err[-1500:])
    _baseline = _torch_smoke_test()
    print(f'[STATUS] Post-repair smoke test: {_baseline}')
    if _baseline.get('cuda_op_ok') is True:
        _repaired = True
        print('[SUCCESS] Repair worked -- proceeding with the cu121 build for the rest of this session.')
    else:
        _diag2 = _diagnose_arch_mismatch(_baseline)
        print('[ERROR] Repair did not fix it.', (_diag2 or ''))
        print('        This looks like a Kaggle platform/accelerator issue rather than something')
        print('        pip can fix from inside the notebook. Try: Session -> Factory reset, or')
        print('        switch the accelerator (Settings -> Accelerator) to a different GPU option')
        print('        and back, then Restart Kernel and re-run from the top. Do not proceed to')
        print('        load any model on GPU until this smoke test reports cuda_op_ok=True.')

# Pin torch/torchvision/torchaudio/triton to whatever is on disk RIGHT NOW (post-repair, if a
# repair happened above), so nothing pulled in as a transitive dependency of ultralytics / CLIP /
# perception_models / the transformers stack below is allowed to silently swap them again.
_CONSTRAINTS_PATH = '/kaggle/working/_torch_constraints.txt'
_pinned = []
for _pkg in ('torch', 'torchvision', 'torchaudio', 'triton'):
    _rc, _out, _err = _sh([sys.executable, '-m', 'pip', 'show', _pkg])
    if _rc == 0:
        for _line in _out.splitlines():
            if _line.startswith('Version:'):
                _pinned.append(f"{_pkg}=={_line.split(':', 1)[1].strip()}")
with open(_CONSTRAINTS_PATH, 'w') as f:
    f.write('\n'.join(_pinned) + '\n')
print(f'[STATUS] Pinning during install: {_pinned}')


def pip_install(args, label):
    result = subprocess.run(
        ['pip', 'install', '-q', '-c', _CONSTRAINTS_PATH] + args,
        capture_output=True, text=True)
    print(f'[STATUS] {label} install exit code: {result.returncode}')
    if result.returncode != 0:
        # A constraint conflict means this package genuinely needs a different
        # torch than what's on disk. Surface it now and stop -- silently retrying
        # without -c would defeat the whole point of pinning.
        print(result.stderr[-2000:])
    return result.returncode


pip_install(['ultralytics'], 'ultralytics (YOLOv8)')
pip_install(['git+https://github.com/openai/CLIP.git'], 'CLIP')

result = subprocess.run(
    ['git', 'clone', 'https://github.com/facebookresearch/perception_models.git'],
    capture_output=True, text=True
)
print('[STATUS] perception_models clone exit code:', result.returncode)
pip_install(['-e', 'perception_models'], 'perception_models')

# -U is not optional. Kaggle ships a transformers that predates Qwen3-VL, and
# `pip install transformers` on an already-satisfied requirement is a no-op --
# it exits 0, prints nothing, and leaves the old version in place. That is
# exactly how a previous run reported "install exit code: 0" and then failed
# with "Transformers does not recognize qwen3_vl", silently fell back to
# constants for all 2027 clips, and produced a submission that scored nothing.
pip_install(['-U', 'transformers>=4.57', 'accelerate', 'qwen-vl-utils', 'bitsandbytes'],
            'transformers stack (upgraded for Qwen3-VL)')

print('[STATUS] Post-install torch smoke test...')
_after = _torch_smoke_test()
print(f'[STATUS] After installs: {_after}')

if _baseline.get('version') != _after.get('version'):
    print(f"[ERROR] torch was changed by the installs above despite the -c constraints "
          f"file: {_baseline.get('version')} -> {_after.get('version')}. One of the "
          f"packages above must have forced this through some path the constraints file "
          f"doesn't cover. Do not proceed to load models until this is understood -- "
          f"Restart Kernel and re-run first.")
elif _after.get('cuda_op_ok') is False:
    print(f"[ERROR] torch version is unchanged ({_after.get('version')}) but a real CUDA "
          f"op now fails: {_after.get('cuda_op_error')}. Since the pre-install baseline "
          f"above already {'was fixed by the repair' if _repaired else 'had the same problem'}, "
          f"this is consistent with an unstable environment rather than these installs -- "
          f"do not proceed to Section 6.10 (Qwen3-VL) or CLIP loading below.")
elif _after.get('cuda_op_ok') is True:
    print('[STATUS] torch unchanged (or successfully repaired) and a real CUDA op still '
          'works -- safe to build on.')

import transformers
from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES

print(f'[STATUS] transformers {transformers.__version__}')
QWEN3_VL_OK = 'qwen3_vl' in CONFIG_MAPPING_NAMES
print(f'[STATUS] qwen3_vl architecture recognized: {QWEN3_VL_OK}')
if not QWEN3_VL_OK:
    print('[FATAL] This transformers cannot load Qwen3-VL. Section 6.10 will not run.')
    print('        Restart the kernel after this cell (an upgrade does not take effect')
    print('        in an already-imported module), or install from source:')
    print('          pip install -U git+https://github.com/huggingface/transformers.git')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[STATUS] torch {torch.__version__} | device: {DEVICE}')


[STATUS] Baseline torch smoke test (before any installs)...
[STATUS] Baseline: {'version': '2.10.0+cu128', 'cuda_available': True, 'device_name': 'Tesla T4', 'device_capability': [7, 5], 'torch_arch_list': ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120'], 'cuda_op_ok': True}
[STATUS] Pinning during install: ['torch==2.10.0+cu128', 'torchvision==0.25.0+cu128', 'torchaudio==2.10.0+cu128', 'triton==3.6.0']
[STATUS] ultralytics (YOLOv8) install exit code: 0
[STATUS] CLIP install exit code: 0
[STATUS] perception_models clone exit code: 0
[STATUS] perception_models install exit code: 0
[STATUS] transformers stack (upgraded for Qwen3-VL) install exit code: 0
[STATUS] Post-install torch smoke test...
[STATUS] After installs: {'version': '2.10.0+cu128', 'cuda_available': True, 'device_name': 'Tesla T4', 'device_capability': [7, 5], 'torch_arch_list': ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120'], 'cuda_op_ok': True}
[STATUS] torch unchanged (or successfully 

---
## 1. Data Acquisition

This section loads all structured metadata from disk: the synthetic labels index, per-video annotation references, and the real test metadata. Raw binary video assets are accessed later via path references rather than bulk loading.

In [6]:
# [LOAD] Synthetic labels index -- primary supervision signal for pipeline calibration
labels_df = pd.read_csv(LABELS_CSV) if LABELS_CSV.exists() else pd.DataFrame()

# [LOAD] Real test metadata -- coarse scene tags provided for analysis, not scoring
test_df = pd.read_csv(TEST_METADATA_CSV) if TEST_METADATA_CSV.exists() else pd.DataFrame()

dim_report = pd.DataFrame({
    'dataset' : ['synthetic_labels', 'test_metadata'],
    'rows'    : [len(labels_df), len(test_df)],
    'columns' : [labels_df.shape[1] if not labels_df.empty else 0,
                 test_df.shape[1] if not test_df.empty else 0],
})
display(dim_report)
print('[SUCCESS] Metadata loaded')

,dataset,rows,columns
0,synthetic_labels,2211,19
1,test_metadata,2027,10


[SUCCESS] Metadata loaded


In [7]:
# [LOAD] Sample submission -- defines expected output schema for the final CSV
sample_sub = pd.read_csv(SAMPLE_SUBMISSION) if SAMPLE_SUBMISSION.exists() else pd.DataFrame()

print('Submission columns:', list(sample_sub.columns))
display(sample_sub.head(3))

Submission columns: []


""


In [8]:
# [LOAD] Enumerate video files
# Synthetic structure: sim_dataset/videos/{head-on,rear-end,sideswipe,single,t-bone}/*.mp4
synthetic_videos = []
if SYNTHETIC_VIDEOS_DIR.exists():
    for subdir in COLLISION_TYPES:
        synthetic_videos.extend(sorted((SYNTHETIC_VIDEOS_DIR / subdir).glob('*.mp4')))

# Real CCTV test videos: flat directory under videos/
real_videos = sorted(REAL_VIDEOS_DIR.glob('*.mp4')) if REAL_VIDEOS_DIR.exists() else []

inventory_rows = [{'split': 'real_test', 'collision_type': 'all', 'count': len(real_videos)}]
for subdir in COLLISION_TYPES:
    sp = SYNTHETIC_VIDEOS_DIR / subdir
    inventory_rows.append({
        'split': 'synthetic', 'collision_type': subdir,
        'count': len(list(sp.glob('*.mp4'))) if sp.exists() else 0,
    })

display(pd.DataFrame(inventory_rows))
print(f'[STATUS] Total synthetic videos : {len(synthetic_videos)}')
print(f'[STATUS] Total real test videos : {len(real_videos)}')

,split,collision_type,count
0,real_test,all,2027
1,synthetic,head-on,588
2,synthetic,rear-end,794
3,synthetic,sideswipe,405
4,synthetic,single,66
5,synthetic,t-bone,358


[STATUS] Total synthetic videos : 2211
[STATUS] Total real test videos : 2027


In [9]:
# [LOAD] Annotation files inventory
# video_annotations/ contains subdirs named like Town03_head-on_clear_00.json/
# Each subdir holds the actual annotation file(s) -- recurse and filter to files only
annotation_files = [
    p for p in sorted(VIDEO_ANNOTATIONS_DIR.rglob('*')) if p.is_file()
] if VIDEO_ANNOTATIONS_DIR.exists() else []

ann_subdirs = [
    p for p in sorted(VIDEO_ANNOTATIONS_DIR.glob('*')) if p.is_dir()
] if VIDEO_ANNOTATIONS_DIR.exists() else []

display(pd.DataFrame({
    'metric' : ['annotation_dir_exists', 'annotation_subdirs', 'annotation_files', 'sample_subdir_name'],
    'value'  : [VIDEO_ANNOTATIONS_DIR.exists(), len(ann_subdirs), len(annotation_files),
                ann_subdirs[0].name if ann_subdirs else 'n/a'],
}))

# [LOAD] Inspect schema of the first annotation file (handles .json and .json.gz)
if annotation_files:
    first_ann = annotation_files[0]
    open_fn = gzip.open if first_ann.suffix == '.gz' else open
    with open_fn(first_ann, 'rt', encoding='utf-8') as f:
        ann_sample = json.load(f)
    top_keys = list(ann_sample.keys()) if isinstance(ann_sample, dict) else f'list[{len(ann_sample)}]'
    print(f'[STATUS] {first_ann.name} top-level keys:', top_keys)
else:
    print('[STATUS] No annotation files found')

,metric,value
0,annotation_dir_exists,True
1,annotation_subdirs,2211
2,annotation_files,2211
3,sample_subdir_name,Town03_head-on_clear_00.json


[STATUS] Town03_head-on_clear_00.json top-level keys: ['base', 'collision', 'sensor']


---
## 2. Data Inspection

Systematic audit of all loaded data structures: schema review, null quantification, statistical profiling, and video-level properties (resolution, FPS, duration) extracted from a sample of clips for hardware-aware planning.

In [10]:
# [INSPECT] Schema of synthetic labels -- column names, dtypes, null counts
if not labels_df.empty:
    display(pd.DataFrame({
        'column'    : labels_df.columns,
        'dtype'     : labels_df.dtypes.values,
        'null_count': labels_df.isnull().sum().values,
        'null_pct'  : (labels_df.isnull().mean().values * 100).round(2),
    }))

,column,dtype,null_count,null_pct
0,rgb_path,object,0,0.0
1,annotations_path,object,0,0.0
2,type,object,0,0.0
3,accident_time,float64,0,0.0
4,accident_frame,int64,0,0.0
5,center_x,float64,0,0.0
6,center_y,float64,0,0.0
7,x1,float64,0,0.0
8,y1,float64,0,0.0
9,x2,float64,0,0.0


In [11]:
# [INSPECT] Statistical summary and sample rows
if not labels_df.empty:
    display(labels_df.describe().T.round(4))
    display(labels_df.head(5))

,count,mean,std,min,25%,50%,75%,max
accident_time,2211.0,7.6057,3.3008,1.0500,5.1500,6.9000,9.8000,20.7500
accident_frame,2211.0,152.1149,66.0161,21.0000,103.0000,138.0000,196.0000,415.0000
center_x,2211.0,0.4984,0.1298,0.0552,0.4186,0.5039,0.5836,0.9305
center_y,2211.0,0.4997,0.1808,0.0176,0.3787,0.5042,0.6241,0.9444
x1,2211.0,0.4508,0.1324,0.0000,0.3693,0.4625,0.5380,0.8615
y1,2211.0,0.4321,0.1720,0.0000,0.3148,0.4389,0.5519,0.8898
x2,2211.0,0.5460,0.1307,0.1104,0.4641,0.5453,0.6307,0.9995
y2,2211.0,0.5674,0.1947,0.0352,0.4333,0.5657,0.6935,0.9991
camera_position,2211.0,45.5563,57.1979,0.0000,14.0000,29.0000,44.5000,240.0000
no_frames,2211.0,353.3641,77.6569,116.0000,299.0000,351.0000,405.0000,644.0000


,rgb_path,annotations_path,type,accident_time,accident_frame,center_x,center_y,x1,y1,x2,y2,map,weather,camera_position,no_frames,duration,height,width,annotations_start_offset
0,videos/sideswipe/Town05_sideswipe_rain_44.mp4,video_annotations/Town05_sideswipe_rain_44.jso...,sideswipe,9.55,191,0.549219,0.387037,0.514583,0.350000,0.583854,0.424074,Town05,rain,44,391,19.55,1080,1920,31
1,videos/sideswipe/Town05_sideswipe_clear_00.mp4,video_annotations/Town05_sideswipe_clear_00.js...,sideswipe,8.65,173,0.494010,0.679167,0.453125,0.595370,0.534896,0.762963,Town05,clear,0,416,20.80,1080,1920,50
2,videos/sideswipe/Town05_sideswipe_sunset_03.mp4,video_annotations/Town05_sideswipe_sunset_03.j...,sideswipe,10.00,200,0.569531,0.890278,0.474479,0.781481,0.664583,0.999074,Town05,sunset,3,407,20.35,1080,1920,40
3,videos/sideswipe/Town05_sideswipe_night_30.mp4,video_annotations/Town05_sideswipe_night_30.js...,sideswipe,7.75,155,0.427604,0.600000,0.393229,0.551852,0.461979,0.648148,Town05,night,30,488,24.40,1080,1920,68
4,videos/sideswipe/Town05_sideswipe_clear_26.mp4,video_annotations/Town05_sideswipe_clear_26.js...,sideswipe,9.40,188,0.579948,0.261111,0.541146,0.195370,0.618750,0.326852,Town05,clear,26,468,23.40,1080,1920,33


In [12]:
# [INSPECT] Collision type distribution in synthetic labels
if not labels_df.empty and 'type' in labels_df.columns:
    type_counts = labels_df['type'].value_counts().reset_index()
    type_counts.columns = ['collision_type', 'count']
    type_counts['pct'] = (type_counts['count'] / type_counts['count'].sum() * 100).round(2)
    display(type_counts)

,collision_type,count,pct
0,rear-end,794,35.91
1,head-on,588,26.59
2,sideswipe,405,18.32
3,t-bone,358,16.19
4,single,66,2.99


In [13]:
# [INSPECT] Schema of real test metadata
if not test_df.empty:
    display(pd.DataFrame({
        'column'    : test_df.columns,
        'dtype'     : test_df.dtypes.values,
        'null_count': test_df.isnull().sum().values,
    }))
    display(test_df.head(5))

,column,dtype,null_count
0,path,object,0
1,region,object,0
2,scene_layout,object,0
3,weather,object,0
4,day_time,object,0
5,quality,object,0
6,no_frames,int64,0
7,duration,float64,0
8,height,int64,0
9,width,int64,0


,path,region,scene_layout,weather,day_time,quality,no_frames,duration,height,width
0,videos/Z4kg2Ev3vhk_00.mp4,Georgia,highway,normal,night,Very_Poor,424,29.967,1080,1584
1,videos/unS0-TLF1ao_00.mp4,World,signalized_intersection,normal,night,Very_Poor,148,5.920,1080,1920
2,videos/UarP8qU1S-c_00.mp4,Virginia,highway,normal,night,Poor,222,15.333,2160,3840
3,videos/UarP8qU1S-c_01.mp4,Virginia,highway,normal,night,Poor,92,6.167,2160,3840
4,videos/nAXTthLfgtI_00.mp4,Virginia,highway,normal,night,Very_Poor,208,13.875,720,960


In [14]:
# [INSPECT] Video-level property extraction from a sample of synthetic clips
# Reads container metadata only -- no full video decoding

def extract_video_meta(video_path: pathlib.Path) -> dict:
    """Return FPS, frame count, width, height, duration for a single video file."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {}
    fps      = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return {
        'path'      : video_path.name,
        'fps'       : round(fps, 2),
        'n_frames'  : n_frames,
        'width'     : width,
        'height'    : height,
        'duration_s': round(n_frames / fps, 2) if fps > 0 else 0.0,
    }

SAMPLE_N = min(20, len(synthetic_videos))
video_meta_df = pd.DataFrame(
    [r for r in (extract_video_meta(p) for p in synthetic_videos[:SAMPLE_N]) if r]
)

if not video_meta_df.empty:
    display(video_meta_df.describe().T.round(2))
else:
    print('[STATUS] No video metadata extracted -- check synthetic video paths')

,count,mean,std,min,25%,50%,75%,max
fps,20.0,20.00,0.00,20.00,20.00,20.0,20.00,20.00
n_frames,20.0,364.75,67.14,249.00,315.00,380.0,419.50,453.00
width,20.0,1920.00,0.00,1920.00,1920.00,1920.0,1920.00,1920.00
height,20.0,1080.00,0.00,1080.00,1080.00,1080.0,1080.00,1080.00
duration_s,20.0,18.24,3.36,12.45,15.75,19.0,20.98,22.65


---
## 3. Data Cleaning

Targeted cleaning of the synthetic labels index: duplicate removal, type-label normalization, out-of-bounds coordinate clamping, and target validation. Video paths are resolved with `resolve_video_path` (which checks both candidate roots) so every retained record points at a file that actually exists.

### 6.1 Model Loading

Two models load by default -- YOLOv8s (~0.5 GB, tracking) and CLIP ViT-B/32 (~0.6 GB, baseline classification fallback). `LOAD_LEGACY_MODELS = False` (Section 6.1 switches) keeps the Perception Encoder, Qwen2.5-VL-3B, and OWLv2 off by default -- Section 7b measured all three losing to a constant predictor, and the code that called them has been removed from the notebook (previously ~4.9 GB loaded for signals nothing used). Their measured numbers are quoted in Section 6.1's model-switch cell and in Sections 6.4 / 6.6 below, kept as documented negative results.


In [32]:
# [MODEL] Model loading switches
#
# The VRAM audit measured 5.40 GB held by the five original models before
# Qwen3-VL-8B even started loading -- which left ~3.8 GB of a 16 GB T4 for
# activations and produced an OOM on the first clip. Qwen3-VL itself is only
# ~6.8 GB in 4-bit; the rest was models nothing calls any more.
#
# Section 7b measured each of them against its constant: the Perception Encoder
# (T=0.31 vs 0.38), OWLv2/optical-flow (S=0.15 vs 0.22), and Qwen2.5-VL-3B
# (C=0.15, below the 0.20 chance level) all lost. The loading code and the
# functions that called them (Stage 1b, 2b, 2c-ensemble, 3b) have been removed
# from the notebook -- see the correction notes in Sections 6.4 and 6.6. Only
# their measured numbers are kept, here and in those two sections.
LOAD_YOLO = True             # still used by Section 7's comparison, ~0.5 GB
LOAD_CLIP = True             # still used by Section 7's comparison, ~0.6 GB

# [MODEL] YOLOv8 -- fast per-frame vehicle detector (primary tracking signal)
YOLO_AVAILABLE = True
try:
    from ultralytics import YOLO
    yolo_model = YOLO('yolov8s.pt')   # COCO-pretrained; small = good speed/accuracy balance
    print('[SUCCESS] YOLOv8s loaded')
except Exception as e:
    YOLO_AVAILABLE = False
    print(f'[ERROR] YOLOv8 unavailable: {e}')

# COCO class ids for vehicles: car, motorcycle, bus, truck
VEHICLE_CLASS_IDS = {2, 3, 5, 7}

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[SUCCESS] YOLOv8s loaded


In [33]:
# [MODEL] CLIP ViT-B/32 -- zero-shot classification backbone
try:
    import clip
    clip_model, clip_preprocess = clip.load('ViT-B/32', device=DEVICE)
    clip_model.eval()
    CLIP_AVAILABLE = True
except Exception as e:
    CLIP_AVAILABLE = False
    print(f'[ERROR] CLIP unavailable: {e}')

if CLIP_AVAILABLE:
    # Pre-encode all collision-type text prompts once.
    # Following the CLIP paper's prompt-ensembling recipe, the per-type mean
    # embedding is RE-NORMALIZED after averaging. Without this, types whose
    # prompts happen to cluster tightly get a systematically larger-norm mean
    # vector and therefore inflated similarity scores.
    TYPE_TEXT_FEATURES = {}
    with torch.no_grad():
        for ctype, prompts in COLLISION_PROMPTS.items():
            tokens   = clip.tokenize(prompts).to(DEVICE)
            features = clip_model.encode_text(tokens)                    # (n_prompts, 512)
            features = features / features.norm(dim=-1, keepdim=True)
            mean_feat = features.mean(dim=0)
            TYPE_TEXT_FEATURES[ctype] = mean_feat / mean_feat.norm()
    print(f'[SUCCESS] CLIP ViT-B/32 loaded | text features for {len(TYPE_TEXT_FEATURES)} types pre-computed')

100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 246MiB/s]


[SUCCESS] CLIP ViT-B/32 loaded | text features for 5 types pre-computed


In [34]:
# [MODEL] VRAM audit -- the two loaded models (YOLOv8s + CLIP) must fit a single T4 (16 GB)
if torch.cuda.is_available():
    print(f'[STATUS] GPU: {torch.cuda.get_device_name(0)}')
    print(f'[STATUS] VRAM allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB | '
          f'reserved: {torch.cuda.memory_reserved(0)/1e9:.2f} GB | '
          f'total: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB')
else:
    print('[STATUS] No CUDA device -- everything will run on CPU (very slow)')

[STATUS] GPU: Tesla T4
[STATUS] VRAM allocated: 0.39 GB | reserved: 0.43 GB | total: 15.64 GB


### 6.2 Vehicle Detection and Tracking (SORT-lite)

Detections are associated frame-to-frame with a minimal SORT-style tracker. The mathematical components:

**Association (Hungarian assignment).** Given predicted track boxes $\hat{B}_i$ and detections $D_j$, solve the linear assignment problem

$$\min_{x_{ij}} \sum_{i,j} c_{ij}\, x_{ij}, \qquad c_{ij} = 1 - \mathrm{IoU}(\hat{B}_i, D_j), \qquad \mathrm{IoU}(A,B) = \frac{|A \cap B|}{|A \cup B|}$$

via the Kuhn-Munkres algorithm (`scipy.optimize.linear_sum_assignment`), rejecting matches below an IoU gate.

**Motion prediction (constant velocity).** Each track's next box is its last box translated by the mean of its recent center displacements -- a zeroth-order Kalman surrogate that is adequate at 10 Hz sampling.

**Velocity estimation (central differences on smoothed centers).** After moving-average smoothing of the center sequence $c_i$, per-sample velocity uses the nonuniform central difference

$$v_i = \frac{c_{i+1} - c_{i-1}}{t_{i+1} - t_{i-1}}$$

(`np.gradient`), which is second-order accurate and does not phase-shift the estimate the way a forward difference does.

In [35]:
# [TRACK] IoU, box gap, and a SORT-style tracker with two-stage association
from scipy.optimize import linear_sum_assignment


def iou_xyxy(a, b) -> float:
    """Intersection-over-union of two [x1, y1, x2, y2] boxes."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter + 1e-9)


def box_gap(a, b) -> float:
    """Euclidean separation between two boxes (0 when they touch or overlap)."""
    gx = max(0.0, max(a[0] - b[2], b[0] - a[2]))
    gy = max(0.0, max(a[1] - b[3], b[1] - a[3]))
    return float(np.hypot(gx, gy))


class SortLiteTracker:
    """Minimal SORT-style multi-object tracker with ByteTrack-style association.

    Constant-velocity prediction on box centers + Hungarian assignment with an
    IoU gate. CCTV cameras are static and vehicles move smoothly, so a full
    Kalman filter adds little at 10 Hz sampling.

    Association runs in two passes over confidence-split detections. This is the
    one idea from ByteTrack that matters for this task: at the moment of impact
    the vehicles occlude and deform, detector confidence collapses, and a
    single-pass tracker with a hard confidence floor drops both tracks at
    exactly the frame the whole pipeline is trying to measure. The second pass
    sustains existing tracks from low-confidence boxes; those boxes are never
    allowed to spawn new tracks, so the noise does not leak in.
    """

    def __init__(self, iou_gate: float = 0.30, iou_gate_low: float = 0.20,
                 max_missed: int = 10, conf_high: float = 0.50):
        self.iou_gate     = iou_gate
        self.iou_gate_low = iou_gate_low
        self.max_missed   = max_missed   # 1.0 s at 10 Hz: a crash occludes for longer
        self.conf_high    = conf_high    # than the old 0.5 s, which split tracks in two
        self.tracks       = {}   # id -> {'boxes': [], 'frames': [], 'confs': [], 'missed': int}
        self.finished     = {}
        self._next_id     = 0

    def _predict(self, tr):
        """Last box translated by the mean of the last <=3 center displacements."""
        boxes = tr['boxes']
        if len(boxes) < 2:
            return boxes[-1]
        centers = np.array([[(b[0] + b[2]) / 2, (b[1] + b[3]) / 2] for b in boxes[-4:]])
        d = np.diff(centers, axis=0).mean(axis=0)
        b = boxes[-1]
        return [b[0] + d[0], b[1] + d[1], b[2] + d[0], b[3] + d[1]]

    def _match(self, tids, detections, gate):
        """Hungarian match of track ids to detections above an IoU gate.

        Returns (pairs, unmatched_track_ids, unmatched_det_indices).
        """
        if not tids or not detections:
            return [], set(tids), set(range(len(detections)))

        cost = np.ones((len(tids), len(detections)))
        for i, tid in enumerate(tids):
            pred = self._predict(self.tracks[tid])
            for j, (box, _) in enumerate(detections):
                cost[i, j] = 1.0 - iou_xyxy(pred, box)

        rows, cols = linear_sum_assignment(cost)
        pairs, matched_t, matched_d = [], set(), set()
        for i, j in zip(rows, cols):
            if 1.0 - cost[i, j] >= gate:
                pairs.append((tids[i], j))
                matched_t.add(tids[i])
                matched_d.add(j)
        return pairs, set(tids) - matched_t, set(range(len(detections))) - matched_d

    def _append(self, tid, det, frame_idx):
        box, conf = det
        tr = self.tracks[tid]
        tr['boxes'].append(list(box))
        tr['frames'].append(frame_idx)
        tr['confs'].append(conf)
        tr['missed'] = 0

    def update(self, detections, frame_idx):
        """detections: list of ([x1,y1,x2,y2], conf) for one sampled frame."""
        high = [d for d in detections if d[1] >= self.conf_high]
        low  = [d for d in detections if d[1] <  self.conf_high]

        # Pass 1: confident detections, strict gate.
        pairs_h, unmatched_t, unmatched_h = self._match(
            list(self.tracks.keys()), high, self.iou_gate)
        for tid, j in pairs_h:
            self._append(tid, high[j], frame_idx)

        # Pass 2: whatever is left gets a shot at the low-confidence boxes.
        pairs_l, unmatched_t, _ = self._match(
            sorted(unmatched_t), low, self.iou_gate_low)
        for tid, j in pairs_l:
            self._append(tid, low[j], frame_idx)

        # Age out tracks that matched nothing in either pass.
        for tid in sorted(unmatched_t):
            self.tracks[tid]['missed'] += 1
            if self.tracks[tid]['missed'] > self.max_missed:
                self.finished[tid] = self.tracks.pop(tid)

        # New tracks spawn from confident detections only.
        for j in sorted(unmatched_h):
            box, conf = high[j]
            self.tracks[self._next_id] = {
                'boxes': [list(box)], 'frames': [frame_idx],
                'confs': [conf], 'missed': 0,
            }
            self._next_id += 1

    def all_tracks(self, min_len: int = 5) -> dict:
        """All tracks (finished + live) with at least min_len observations."""
        out = {}
        for tid, tr in {**self.finished, **self.tracks}.items():
            if len(tr['frames']) >= min_len:
                out[tid] = {
                    'boxes' : np.array(tr['boxes'], dtype=np.float32),
                    'frames': np.array(tr['frames'], dtype=np.int64),
                    'confs' : np.array(tr['confs'], dtype=np.float32),
                }
        return out


print('[STATUS] SortLiteTracker defined (two-stage association)')

[STATUS] SortLiteTracker defined (two-stage association)


In [36]:
# [TRACK] Run detection + tracking over a video, then compute per-track kinematics

from scipy.signal import savgol_filter

# Two vehicles at the moment of impact overlap heavily -- often past the default
# NMS threshold of 0.7, which then deletes one of them as a duplicate box. That
# removes the collision pair from the exact frame the pipeline exists to find.
# 0.9 keeps both; the cost is a few extra duplicate boxes elsewhere, which the
# tracker's IoU gate absorbs.
YOLO_NMS_IOU  = 0.9
YOLO_CONF_MIN = 0.10   # floor for entering the tracker at all; the tracker's
                       # own conf_high splits high/low internally


def extract_vehicle_tracks(video_path: pathlib.Path,
                           sample_fps: float = 10.0,
                           conf_thresh: float = YOLO_CONF_MIN,
                           min_track_len: int = 5):
    """Detect vehicles on frames sampled at sample_fps and link them into tracks.

    Returns (tracks, fps, (W, H)) where tracks maps id -> arrays of boxes
    (pixels), frame indices, and detection confidences.
    """
    cap = cv2.VideoCapture(str(video_path))
    fps      = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if fps <= 0 or n_frames == 0:
        cap.release()
        return {}, fps, (W, H)

    step = max(1, int(round(fps / sample_fps)))
    tracker = SortLiteTracker()

    # Sequential decode (no seeking) -- much faster than per-frame cap.set
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % step == 0:
            results = yolo_model(frame, iou=YOLO_NMS_IOU, conf=conf_thresh,
                                 verbose=False)[0]
            detections = []
            for box, cls, conf in zip(results.boxes.xyxy.cpu().numpy(),
                                      results.boxes.cls.cpu().numpy(),
                                      results.boxes.conf.cpu().numpy()):
                if int(cls) in VEHICLE_CLASS_IDS:
                    detections.append((box.tolist(), float(conf)))
            tracker.update(detections, frame_idx)
        frame_idx += 1

    cap.release()
    return tracker.all_tracks(min_len=min_track_len), fps, (W, H)


def track_kinematics(track: dict, fps: float, smooth_window: int = 5) -> dict:
    """Smoothed center trajectory and central-difference velocity for one track."""
    boxes = track['boxes']
    t  = track['frames'] / fps
    cx = (boxes[:, 0] + boxes[:, 2]) / 2
    cy = (boxes[:, 1] + boxes[:, 3]) / 2

    if len(cx) >= smooth_window:
        # Savitzky-Golay, NOT np.convolve(mode='same') and not a rolling mean.
        # Both distort the ends of the trajectory: convolve zero-pads, dragging
        # the first and last samples hundreds of pixels toward the origin, and a
        # centred rolling mean shrinks them by averaging over a truncated window.
        # Tracks typically END at the collision (the vehicle stops, the tracker
        # loses it) and _impact_time searches for the peak joint deceleration --
        # so an edge artifact lands precisely on the signal being measured and
        # fabricates an impact. savgol with mode='interp' fits a polynomial to
        # the edge samples instead of inventing data, and reproduces a constant
        # velocity exactly (measured: 0.0 px/s error, against 50 for a rolling
        # mean and 680 for convolve).
        #
        # savgol assumes uniform spacing; a track with missed frames is not
        # strictly uniform. The velocity below therefore still uses np.gradient
        # against the real timestamps, and the residual error at a gap is
        # unbiased rather than systematic.
        cx = savgol_filter(cx, smooth_window, 2, mode='interp')
        cy = savgol_filter(cy, smooth_window, 2, mode='interp')

    vx = np.gradient(cx, t)
    vy = np.gradient(cy, t)
    return {
        't': t, 'cx': cx, 'cy': cy, 'vx': vx, 'vy': vy,
        'speed': np.hypot(vx, vy),
        'frames': track['frames'], 'boxes': boxes,
        'conf': float(np.median(track['confs'])),
    }


print('[STATUS] extract_vehicle_tracks / track_kinematics defined')

[STATUS] extract_vehicle_tracks / track_kinematics defined


### 6.3 Collision Analysis from Track Kinematics

**Contact detection.** For each pair of concurrently tracked vehicles, the box gap $g(t)$ is monitored on their shared time support; contact is the first sample where $g(t)$ falls below 1% of the frame diagonal.

**Impact time refinement.** Within $\pm0.6$ s of contact, the impact instant is the peak of joint deceleration

$$t^{*} = \arg\max_{t} \left[ -\frac{d}{dt}\left( \lVert v_A(t) \rVert + \lVert v_B(t) \rVert \right) \right]$$

-- physically, the moment momentum is exchanged.

**Impact point.** The centroid of the intersection of the two boxes at $t^{*}$ (boxes dilated slightly when they only touch), normalized by frame size.

**Type from geometry.** With pre-impact mean velocities $\bar{v}_A, \bar{v}_B$ over $[t^{*}-1.2, t^{*}-0.2]$ s, the approach angle is

$$\theta = \arccos\left( \frac{\bar{v}_A \cdot \bar{v}_B}{\lVert \bar{v}_A \rVert\, \lVert \bar{v}_B \rVert} \right)$$

and the decision uses $\theta$ plus the contact bearing (is the struck point ahead of, beside, or behind each vehicle relative to its own motion):

| Condition | Type |
|-----------|------|
| $\theta \geq 135^{\circ}$ | head-on |
| $\theta \leq 40^{\circ}$, contact along the leader's motion axis | rear-end |
| $\theta \leq 40^{\circ}$, contact lateral | sideswipe |
| $60^{\circ} \leq \theta \leq 120^{\circ}$ | t-bone |
| otherwise (ambiguous bands) | no hint -- defer to VLM ensemble |

**Single-vehicle crashes** need no partner: a track whose speed collapses by more than 60% within a short window, with no other vehicle in contact, is a `single` candidate.

**Gating.** Geometry is only computed when both tracks are long ($\geq 6$ samples), confident (median detection confidence $\geq 0.4$), and moving ($\lVert \bar{v} \rVert$ above a floor) -- exactly the preconditions whose absence sank the raw-optical-flow version of this idea.

In [45]:
# [CALIB] What tracking is actually worth -- measured on every extracted video
m = features_df[features_df['found']].merge(
    labels_clean[['rgb_path', 'accident_time', 'center_x', 'center_y']],
    on='rgb_path', how='inner')
print(f'[STATUS] Measuring on {len(m)} videos where tracking fired')

# A tracked contact is the instant the BOXES touch. The dataset's accident_time
# may be defined differently -- on a synthetic t-bone the boxes overlap ~0.5 s
# before the centres coincide, and any such systematic offset is free score on T.
# Measured on the training set and subtracted; this is calibration on provided
# labels, not on the test set.
resid = (m['t_track'] - m['accident_time']).to_numpy()
TIME_OFFSET = float(np.median(resid))
print(f'[CALIB] t_track - accident_time: median={TIME_OFFSET:+.3f}s  '
      f'IQR=[{np.quantile(resid, .25):+.3f}, {np.quantile(resid, .75):+.3f}]')

def _Tm(p, g): return float(np.exp(-0.5 * ((p - g) / SIGMA_T) ** 2).mean())
def _Sm(px, py, gx, gy):
    return float(np.exp(-0.5 * ((px - gx) ** 2 + (py - gy) ** 2) / SIGMA_S ** 2).mean())

gt_t, gt_x, gt_y = m['accident_time'].to_numpy(), m['center_x'].to_numpy(), m['center_y'].to_numpy()
cov = float(features_df['found'].mean())
n_all = len(features_df)

rows = [
    ('T  constant',            _Tm(np.full(len(m), CONST['accident_time']), gt_t), 1.0),
    ('T  tracking (raw)',      _Tm(m['t_track'].to_numpy(), gt_t), cov),
    ('T  tracking (offset)',   _Tm(m['t_track'].to_numpy() - TIME_OFFSET, gt_t), cov),
    ('S  constant',            _Sm(np.full(len(m), CONST['center_x']),
                                   np.full(len(m), CONST['center_y']), gt_x, gt_y), 1.0),
    ('S  tracking',            _Sm(m['x_track'].to_numpy(), m['y_track'].to_numpy(), gt_x, gt_y), cov),
]
# Constant scores on the same subset, for the coverage-weighted blend below
T_con_all = _Tm(np.full(len(m), CONST['accident_time']), gt_t)
S_con_all = _Sm(np.full(len(m), CONST['center_x']), np.full(len(m), CONST['center_y']), gt_x, gt_y)

out = []
for name, val, c in rows:
    base = T_con_all if name.startswith('T') else S_con_all
    blended = val * c + base * (1 - c) if c < 1.0 else val
    out.append({'signal': name, 'coverage': round(c, 3),
                'score_where_it_fires': round(val, 4),
                'blended_with_constant': round(blended, 4)})
print()
display(pd.DataFrame(out))
print(f'[VERDICT] tracking beats the constant on T: '
      f"{out[2]['blended_with_constant'] > out[0]['score_where_it_fires']}")
print(f'[VERDICT] tracking beats the constant on S: '
      f"{out[4]['blended_with_constant'] > out[3]['score_where_it_fires']}")
print('[NOTE] Measured inside CARLA. It says nothing about whether tracking '
      'fires at all on real CCTV -- only running it on the real videos does.')

[STATUS] Measuring on 1673 videos where tracking fired
[CALIB] t_track - accident_time: median=+0.100s  IQR=[-2.100, +2.950]



,signal,coverage,score_where_it_fires,blended_with_constant
0,T constant,1.000,0.5529,0.5529
1,T tracking (raw),0.757,0.4868,0.5029
2,T tracking (offset),0.757,0.4864,0.5026
3,S constant,1.000,0.3066,0.3066
4,S tracking,0.757,0.3481,0.3380


[VERDICT] tracking beats the constant on T: False
[VERDICT] tracking beats the constant on S: True
[NOTE] Measured inside CARLA. It says nothing about whether tracking fires at all on real CCTV -- only running it on the real videos does.
